# AVeriTeC Advanced Evaluation

Includes evidence-aware diagnostics.

In [1]:
import json
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score
from collections import Counter


In [ ]:
# Load
path = "../outputs/2026-03-21_01-20-48/averitec_dev.json"
with open(path) as f:
    data = json.load(f)

df = pd.DataFrame(data)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: '../outputs/2026-03-01_20-00-48/averitec_dev.json'

In [ ]:
# Normalize labels
def normalize(x):
    return x.lower().replace("/", "").replace(" ", "")

df["gold_norm"] = df["gold_label"].apply(normalize)
df["pred_norm"] = df["prediction"].apply(normalize)


In [ ]:
# Accuracy + report
print("Accuracy:", accuracy_score(df["gold_norm"], df["pred_norm"]))
print(classification_report(df["gold_norm"], df["pred_norm"]))


## 🔍 Evidence Diversity

In [ ]:
def evidence_stats(row):
    texts = []
    for step in row["pipeline"]["steps"]:
        for ev in step.get("evidence", []):
            texts.append(ev["text"])

    unique = len(set(texts))
    total = len(texts)
    return pd.Series({
        "evidence_total": total,
        "evidence_unique": unique,
        "diversity_ratio": unique / total if total > 0 else 0
    })

ev_stats = df.apply(evidence_stats, axis=1)
df = pd.concat([df, ev_stats], axis=1)

df[["evidence_total", "evidence_unique", "diversity_ratio"]].describe()


## 🚨 Duplicate Evidence

In [ ]:
df["duplication_rate"] = 1 - df["diversity_ratio"]
df["duplication_rate"].describe()


## 🎯 AVeriTeC-style proxy score

In [ ]:
# proxy: correct label AND diversity > threshold
df["label_correct"] = df["gold_norm"] == df["pred_norm"]

df["good_evidence"] = df["diversity_ratio"] > 0.5

df["averitec_proxy"] = df["label_correct"] & df["good_evidence"]

print("Proxy score:", df["averitec_proxy"].mean())


## 🔬 Error Breakdown

In [ ]:
conditions = []

for _, row in df.iterrows():
    if not row["label_correct"]:
        conditions.append("label_error")
    elif not row["good_evidence"]:
        conditions.append("evidence_error")
    else:
        conditions.append("correct")

df["error_type"] = conditions

df["error_type"].value_counts()


## 🔎 Inspect worst cases

In [ ]:
df.sort_values("diversity_ratio").head(5)[["claim","diversity_ratio"]]
